# 01. Dataset de modelagem — snapshots de contratos de obra

Este notebook reproduz a construção do arquivo `dataset_modelagem_obras_snapshots.csv`.

Regras principais:

- o universo elegível é definido **exclusivamente por `vw_bi_contratos.csv`**;
- parcelas, liberações e aditivos são apenas fontes auxiliares de features;
- datas sentinela como `1900-01-01` são tratadas como ausentes;
- cada contrato gera snapshots mensais desde a `Data_OS` até a conclusão ou `DATA_REFERENCIA`;
- toda feature é calculada usando somente eventos com data `<= data_corte`;
- aditivos de prazo futuros não alteram retrospectivamente snapshots antigos;
- `target_dias_atraso` e `target_atraso` só são conhecidos para contratos concluídos;
- `flag_apto_treino = 1` somente para snapshots anteriores à conclusão e com target conhecido.

> Para reproduzir exatamente o dataset entregue anteriormente, mantenha `DATA_REFERENCIA = "2026-09-29"` e utilize os mesmos cinco arquivos-fonte.


In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FILES = [
    "vw_bi_contratos.csv",
    "vw_bi_parcelas.csv",
    "vw_tparcela_liberacao.csv",
    "vw_taditivovalor.csv",
    "vw_taditivoprazo.csv",
]

# Data usada na construção do dataset entregue.
# Atualize quando quiser gerar snapshots mais recentes para contratos em andamento.
DATA_REFERENCIA = pd.Timestamp("2026-09-29")


## 1. Pastas e carregamento

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate
    return current

ROOT = find_project_root()

if (ROOT / "data" / "raw").exists():
    RAW = ROOT / "data" / "raw"
    PROCESSED = ROOT / "data" / "processed"
    QUALITY = ROOT / "reports" / "data_quality"
else:
    # Fallback útil quando o notebook e os CSVs estão na mesma pasta.
    RAW = ROOT
    PROCESSED = ROOT / "data" / "processed"
    QUALITY = ROOT / "reports" / "data_quality"

for folder in [PROCESSED, QUALITY]:
    folder.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RAW:", RAW)
print("PROCESSED:", PROCESSED)


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(RAW / name, low_memory=False, encoding="utf-8-sig")


contratos = read_csv("vw_bi_contratos.csv")
parcelas = read_csv("vw_bi_parcelas.csv")
liberacoes = read_csv("vw_tparcela_liberacao.csv")
aditivos_valor = read_csv("vw_taditivovalor.csv")
aditivos_prazo = read_csv("vw_taditivoprazo.csv")

datasets = {
    "contratos": contratos,
    "parcelas": parcelas,
    "liberacoes": liberacoes,
    "aditivos_valor": aditivos_valor,
    "aditivos_prazo": aditivos_prazo,
}

manifest = pd.DataFrame([
    {
        "arquivo": name,
        "linhas": len(datasets[key]),
        "colunas": datasets[key].shape[1],
        "tamanho_bytes": (RAW / name).stat().st_size,
        "sha256": sha256(RAW / name),
    }
    for key, name in zip(datasets, FILES)
])

manifest.to_csv(QUALITY / "manifesto_carga_modelagem.csv", index=False, encoding="utf-8-sig")
display(manifest)


## 2. Padronização de tipos e datas sentinela

In [ ]:
def to_dates(df, columns):
    result = df.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_datetime(result[column], errors="coerce")
            # Datas históricas/sentinela incompatíveis com o domínio são tratadas como missing.
            result.loc[result[column].dt.year <= 1900, column] = pd.NaT
    return result


def to_numbers(df, columns):
    result = df.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    return result


contratos = to_numbers(
    contratos,
    ["Id_contrato", "Valor_total_contrato", "Prazo_total", "Id_fornecedor"],
)
contratos = to_dates(
    contratos,
    [
        "Data_assinatura_contrato", "Data_conclusao_contrato", "Data_conclusao",
        "Data_proposta", "Data_Orcamento", "Data_OS",
        "Data_Paralisacao", "Data_Prev_Retomada",
    ],
)

parcelas = to_numbers(
    parcelas,
    ["id_parcela", "id_contrato", "valor_reajustado", "ordem_status", "valor_ja_liberado"],
)
parcelas = to_dates(
    parcelas,
    ["data_prevista_parcela", "data_efetiva", "data_liberacao"],
)

liberacoes = to_numbers(
    liberacoes,
    ["id_Contrato", "id_Parcela", "Abono_atraso", "Atraso_considerado", "Valor_total_liberado"],
)
liberacoes = to_dates(
    liberacoes,
    ["Data_efet_conclusao", "Data_liberação"],
)

aditivos_valor = to_numbers(aditivos_valor, ["idContrato"])
aditivos_valor = to_dates(aditivos_valor, ["Data_assinatura_TA_valor"])

aditivos_prazo = to_numbers(aditivos_prazo, ["idContrato", "Aditado_em_dias"])
aditivos_prazo = to_dates(
    aditivos_prazo,
    ["Data_termino_anterior", "Data_termino_nova", "Data_assinatura_ta_prazo"],
)


## 3. Universo de contratos de obra

`vw_bi_contratos` define o universo elegível. Registros existentes apenas nas demais fontes são descartados antes da construção das features.


In [ ]:
summary = pd.DataFrame([
    {
        "dataset": name,
        "linhas": len(df),
        "colunas": df.shape[1],
        "duplicidades_exatas": int(df.duplicated().sum()),
        "percentual_nulos": round(float(df.isna().mean().mean() * 100), 2),
    }
    for name, df in {
        "contratos": contratos,
        "parcelas": parcelas,
        "liberacoes": liberacoes,
        "aditivos_valor": aditivos_valor,
        "aditivos_prazo": aditivos_prazo,
    }.items()
])
display(summary)

ids_obras = set(
    pd.to_numeric(contratos["Id_contrato"], errors="coerce").dropna().astype(int)
)

parcelas = parcelas[parcelas["id_contrato"].isin(ids_obras)].copy()
liberacoes = liberacoes[liberacoes["id_Contrato"].isin(ids_obras)].copy()
aditivos_valor = aditivos_valor[aditivos_valor["idContrato"].isin(ids_obras)].copy()
aditivos_prazo = aditivos_prazo[aditivos_prazo["idContrato"].isin(ids_obras)].copy()

print(f"Contratos de obra no cadastro: {len(ids_obras)}")
print(f"Contratos com Data_OS válida: {contratos['Data_OS'].notna().sum()}")


## 4. Calendário mensal de snapshots

Para cada obra com `Data_OS` válida:

1. o primeiro snapshot é a própria `Data_OS`;
2. são incluídos os últimos dias de cada mês;
3. o último snapshot é a data de conclusão, quando já conhecida, ou `DATA_REFERENCIA`;
4. contratos cuja conclusão seja posterior a `DATA_REFERENCIA` são truncados em `DATA_REFERENCIA`.

Esse desenho evita criar snapshots apenas quando há eventos e deixa as trajetórias comparáveis ao longo do tempo.


In [ ]:
calendar_rows = []

for _, contrato in contratos[contratos["Data_OS"].notna()].iterrows():
    inicio = contrato["Data_OS"]

    fim_natural = (
        contrato["Data_conclusao"]
        if pd.notna(contrato["Data_conclusao"])
        else DATA_REFERENCIA
    )
    fim = min(fim_natural, DATA_REFERENCIA)

    if fim < inicio:
        continue

    cortes = list(pd.date_range(inicio, fim, freq="ME"))

    if not cortes or cortes[0] > inicio:
        cortes = [inicio] + cortes

    if cortes[-1] != fim:
        cortes.append(fim)

    for data_corte in sorted(set(cortes)):
        calendar_rows.append((int(contrato["Id_contrato"]), data_corte))

calendar = pd.DataFrame(calendar_rows, columns=["id_contrato", "data_corte"])

assert not calendar.duplicated(["id_contrato", "data_corte"]).any()

print(f"Snapshots no calendário: {len(calendar):,}")
print(f"Contratos no calendário: {calendar['id_contrato'].nunique():,}")
display(calendar.head())


## 5. Features point-in-time e targets

Para cada `id_contrato + data_corte`, as features usam apenas eventos conhecidos até a fotografia.

### Prazo vigente na fotografia
- começa em `Data_OS + Prazo_total`;
- se já houver aditivo de prazo assinado, usa a `Data_termino_nova` do aditivo mais recente conhecido naquele corte.

### Targets
Para contratos concluídos, o prazo final de referência considera somente aditivos assinados até a própria conclusão.

- `target_dias_atraso = max(0, Data_conclusao - data_termino_vigente_final)`
- `target_atraso = 1` se `target_dias_atraso > 0`, senão `0`

Os campos terminados em `_controle` são de auditoria e **não devem entrar em X**.


In [ ]:
contratos_idx = contratos.set_index("Id_contrato")

rows = []

for cid, data_corte in calendar.itertuples(index=False, name=None):
    contrato = contratos_idx.loc[cid]

    # Prazo original
    data_termino_original = (
        contrato["Data_OS"] + pd.to_timedelta(float(contrato["Prazo_total"]), unit="D")
        if pd.notna(contrato["Prazo_total"])
        else pd.NaT
    )

    # Aditivos de prazo conhecidos até o snapshot
    api = aditivos_prazo[
        (aditivos_prazo["idContrato"] == cid)
        & aditivos_prazo["Data_assinatura_ta_prazo"].notna()
        & (aditivos_prazo["Data_assinatura_ta_prazo"] <= data_corte)
    ].sort_values("Data_assinatura_ta_prazo")

    if api["Data_termino_nova"].notna().any():
        data_termino_vigente_corte = api["Data_termino_nova"].dropna().iloc[-1]
    else:
        data_termino_vigente_corte = data_termino_original

    # Parcelas do contrato
    pi = parcelas[parcelas["id_contrato"] == cid]

    parcelas_previstas = pi[
        pi["data_prevista_parcela"].notna()
        & (pi["data_prevista_parcela"] <= data_corte)
    ]

    parcelas_concluidas = pi[
        pi["data_efetiva"].notna()
        & (pi["data_efetiva"] <= data_corte)
    ]

    atrasos = (
        parcelas_concluidas["data_efetiva"]
        - parcelas_concluidas["data_prevista_parcela"]
    ).dt.days.dropna()

    parcelas_90d = parcelas_concluidas[
        parcelas_concluidas["data_efetiva"]
        >= data_corte - pd.Timedelta(days=90)
    ]
    atrasos_90d = (
        parcelas_90d["data_efetiva"]
        - parcelas_90d["data_prevista_parcela"]
    ).dt.days.dropna()

    # Liberações conhecidas até o snapshot
    li = liberacoes[
        (liberacoes["id_Contrato"] == cid)
        & liberacoes["Data_liberação"].notna()
        & (liberacoes["Data_liberação"] <= data_corte)
    ]

    li_90d = li[
        li["Data_liberação"] >= data_corte - pd.Timedelta(days=90)
    ]

    # Aditivos de valor conhecidos até o snapshot
    avi = aditivos_valor[
        (aditivos_valor["idContrato"] == cid)
        & aditivos_valor["Data_assinatura_TA_valor"].notna()
        & (aditivos_valor["Data_assinatura_TA_valor"] <= data_corte)
    ]

    dias_desde_os = (data_corte - contrato["Data_OS"]).days

    prazo_vigente_total_dias = (
        (data_termino_vigente_corte - contrato["Data_OS"]).days
        if pd.notna(data_termino_vigente_corte)
        else np.nan
    )

    pct_prazo_consumido = (
        dias_desde_os / prazo_vigente_total_dias
        if pd.notna(prazo_vigente_total_dias) and prazo_vigente_total_dias > 0
        else np.nan
    )

    valor_total_contrato = pd.to_numeric(
        pd.Series([contrato["Valor_total_contrato"]]), errors="coerce"
    ).iloc[0]

    valor_liberado_acumulado = (
        li["Valor_total_liberado"].sum(min_count=1) if len(li) else 0
    )

    pct_financeiro_liberado = (
        valor_liberado_acumulado / valor_total_contrato
        if pd.notna(valor_total_contrato) and valor_total_contrato != 0
        else np.nan
    )

    # Prazo final usado no target: somente informações formalizadas até a conclusão
    data_conclusao = contrato["Data_conclusao"]

    if pd.notna(data_conclusao):
        aditivos_ate_conclusao = aditivos_prazo[
            (aditivos_prazo["idContrato"] == cid)
            & aditivos_prazo["Data_assinatura_ta_prazo"].notna()
            & (aditivos_prazo["Data_assinatura_ta_prazo"] <= data_conclusao)
        ].sort_values("Data_assinatura_ta_prazo")

        if aditivos_ate_conclusao["Data_termino_nova"].notna().any():
            data_termino_vigente_final = (
                aditivos_ate_conclusao["Data_termino_nova"].dropna().iloc[-1]
            )
        else:
            data_termino_vigente_final = data_termino_original
    else:
        data_termino_vigente_final = pd.NaT

    target_dias_atraso = (
        max(0, (data_conclusao - data_termino_vigente_final).days)
        if pd.notna(data_conclusao) and pd.notna(data_termino_vigente_final)
        else np.nan
    )

    rows.append({
        "id_contrato": cid,
        "data_corte": data_corte,
        "numero_contrato": contrato["Numero_contrato"],
        "classificacao_contrato": contrato["Classificacao_contrato"],
        "id_fornecedor": contrato["Id_fornecedor"],
        "valor_total_contrato": valor_total_contrato,
        "prazo_original_dias": contrato["Prazo_total"],
        "data_os": contrato["Data_OS"],
        "data_termino_original": data_termino_original,
        "data_termino_vigente_corte": data_termino_vigente_corte,
        "dias_desde_os": dias_desde_os,
        "dias_restantes_prazo_vigente": (
            (data_termino_vigente_corte - data_corte).days
            if pd.notna(data_termino_vigente_corte)
            else np.nan
        ),
        "prazo_vigente_total_dias": prazo_vigente_total_dias,
        "pct_prazo_consumido": pct_prazo_consumido,
        "qtd_aditivos_prazo": len(api),
        "dias_aditados_acumulados": (
            api["Aditado_em_dias"].sum(min_count=1) if len(api) else 0
        ),
        "dias_desde_ultimo_aditivo_prazo": (
            (data_corte - api["Data_assinatura_ta_prazo"].max()).days
            if len(api) else np.nan
        ),
        "qtd_aditivos_valor": len(avi),
        "dias_desde_ultimo_aditivo_valor": (
            (data_corte - avi["Data_assinatura_TA_valor"].max()).days
            if len(avi) else np.nan
        ),
        "qtd_parcelas_previstas_ate_corte": len(parcelas_previstas),
        "qtd_parcelas_concluidas_ate_corte": len(parcelas_concluidas),
        "pct_parcelas_concluidas_vs_previstas": (
            len(parcelas_concluidas) / len(parcelas_previstas)
            if len(parcelas_previstas)
            else np.nan
        ),
        "atraso_medio_parcelas_dias": atrasos.mean() if len(atrasos) else np.nan,
        "atraso_mediano_parcelas_dias": atrasos.median() if len(atrasos) else np.nan,
        "atraso_maximo_parcelas_dias": atrasos.max() if len(atrasos) else np.nan,
        "qtd_parcelas_atrasadas": int((atrasos > 0).sum()) if len(atrasos) else 0,
        "pct_parcelas_atrasadas": float((atrasos > 0).mean()) if len(atrasos) else np.nan,
        "atraso_medio_parcelas_90d": atrasos_90d.mean() if len(atrasos_90d) else np.nan,
        "qtd_parcelas_atrasadas_90d": int((atrasos_90d > 0).sum()) if len(atrasos_90d) else 0,
        "qtd_liberacoes": len(li),
        "valor_liberado_acumulado": valor_liberado_acumulado,
        "valor_liberado_90d": (
            li_90d["Valor_total_liberado"].sum(min_count=1) if len(li_90d) else 0
        ),
        "atraso_medio_liberacao_dias": (
            li["Atraso_considerado"].mean() if len(li) else np.nan
        ),
        "atraso_maximo_liberacao_dias": (
            li["Atraso_considerado"].max() if len(li) else np.nan
        ),
        "pct_financeiro_liberado": pct_financeiro_liberado,
        "gap_prazo_financeiro": (
            pct_prazo_consumido - pct_financeiro_liberado
            if pd.notna(pct_prazo_consumido) and pd.notna(pct_financeiro_liberado)
            else np.nan
        ),
        "status_final_controle": contrato["Status_Contrato"],
        "data_conclusao_controle": data_conclusao,
        "data_termino_vigente_final_controle": data_termino_vigente_final,
        "target_dias_atraso": target_dias_atraso,
        "target_atraso": (
            int(target_dias_atraso > 0) if pd.notna(target_dias_atraso) else np.nan
        ),
    })

dataset = pd.DataFrame(rows)
dataset.head()


## 6. Faixa de antecedência, elegibilidade de treino e exportação

In [ ]:
dataset["faixa_antecedencia"] = pd.cut(
    dataset["dias_restantes_prazo_vigente"],
    [-np.inf, 29, 89, 179, np.inf],
    labels=["<30d", "30-89d", "90-179d", ">=180d"],
)

dataset["flag_apto_treino"] = (
    dataset["data_conclusao_controle"].notna()
    & (dataset["data_corte"] < dataset["data_conclusao_controle"])
    & dataset["target_dias_atraso"].notna()
).astype("int8")

dataset = dataset.sort_values(["id_contrato", "data_corte"]).reset_index(drop=True)

OUTPUT = PROCESSED / "dataset_modelagem_obras_snapshots.csv"
dataset.to_csv(OUTPUT, index=False, encoding="utf-8-sig")

print("Arquivo:", OUTPUT)
print(f"Linhas: {len(dataset):,}")
print(f"Colunas: {dataset.shape[1]}")
print(f"Contratos: {dataset['id_contrato'].nunique():,}")
print(f"Snapshots aptos para treino: {int(dataset['flag_apto_treino'].sum()):,}")
print(
    "Contratos com snapshots aptos para treino:",
    dataset.loc[dataset["flag_apto_treino"].eq(1), "id_contrato"].nunique(),
)


## 7. Validações de consistência

As validações abaixo não fixam quantidade de linhas/contratos, para que o notebook continue funcionando quando as fontes forem atualizadas. Com os arquivos usados na geração original e `DATA_REFERENCIA = 2026-09-29`, o resultado de referência é:

- 1.435 snapshots;
- 78 contratos com `Data_OS` válida e intervalo temporal utilizável;
- 43 colunas;
- 1.034 snapshots aptos para treino;
- 59 contratos com snapshots aptos para treino.


In [ ]:
assert not dataset.duplicated(["id_contrato", "data_corte"]).any()
assert set(dataset["id_contrato"]).issubset(ids_obras)

# Nenhuma linha apta a treino pode estar na própria conclusão ou depois dela.
assert (
    dataset.loc[dataset["flag_apto_treino"].eq(1), "data_corte"]
    < dataset.loc[dataset["flag_apto_treino"].eq(1), "data_conclusao_controle"]
).all()

# Contratos sem conclusão não recebem target supervisionado.
sem_conclusao = dataset["data_conclusao_controle"].isna()
assert dataset.loc[sem_conclusao, "target_dias_atraso"].isna().all()
assert dataset.loc[sem_conclusao, "target_atraso"].isna().all()

quality = pd.DataFrame({
    "metrica": [
        "snapshots",
        "colunas",
        "contratos",
        "data_corte_minima",
        "data_corte_maxima",
        "duplicidades_id_data",
        "snapshots_aptos_treino",
        "contratos_aptos_treino",
        "taxa_atraso_snapshots_treino",
    ],
    "valor": [
        len(dataset),
        dataset.shape[1],
        dataset["id_contrato"].nunique(),
        dataset["data_corte"].min(),
        dataset["data_corte"].max(),
        dataset.duplicated(["id_contrato", "data_corte"]).sum(),
        int(dataset["flag_apto_treino"].sum()),
        dataset.loc[dataset["flag_apto_treino"].eq(1), "id_contrato"].nunique(),
        dataset.loc[dataset["flag_apto_treino"].eq(1), "target_atraso"].mean(),
    ],
})

quality.to_csv(
    QUALITY / "qualidade_dataset_modelagem_obras_snapshots.csv",
    index=False,
    encoding="utf-8-sig",
)
display(quality)


## 8. Colunas que não devem entrar diretamente no modelo

Excluir de `X`:

- `id_contrato` e `numero_contrato`: identificadores;
- `status_final_controle`;
- `data_conclusao_controle`;
- `data_termino_vigente_final_controle`;
- `target_dias_atraso`;
- `target_atraso`;
- `flag_apto_treino` (serve apenas para filtro);
- datas absolutas, caso não sejam transformadas em features apropriadas.

`data_corte` deve ser preservada para validação temporal e auditoria, mas não necessariamente usada como preditor bruto.
